# Clase 06 · Demo guiada: embeddings y búsqueda semántica — Gestor de solicitudes

**Proyecto:** Gestor de solicitudes — clasifica peticiones y prepara respuestas según su tipo, prioridad y responsable.

Partimos del clasificador y la consulta de estado de las clases anteriores; hoy añadimos una forma de buscar información en un corpus. Esta demo recupera pasajes, pero todavía no genera respuestas con ellos.

Usaremos cinco textos **ficticios** sobre cómo se atienden las solicitudes. No representan procedimientos reales de ninguna organización. Ejecuta las celdas en orden en Google Colab. La primera ejecución descarga un modelo de embeddings; no se necesita clave de API ni GPU.

## 1. Preparación
Instalamos la biblioteca para generar embeddings. En una sesión nueva de Colab se ejecuta esta celda una vez.

In [8]:
%pip -q install sentence-transformers

## 2. Nuestro corpus
Cada pasaje conserva un identificador y su texto. Pregunta al grupo: **¿cuál de estos pasajes serviría para orientar a alguien cuyo computador no enciende?**

In [2]:
corpus = [
    {"id": "P1", "texto": "Las fallas de acceso, correo o equipos se registran como soporte técnico y las atiende la mesa de ayuda."},
    {"id": "P2", "texto": "Si una solicitud deja a un área sin servicio, se marca con prioridad alta y se atiende antes que las demás."},
    {"id": "P3", "texto": "El estado de una solicitud se consulta en el módulo de seguimiento mediante su código."},
    {"id": "P4", "texto": "Las compras de equipos o suministros se radican con descripción, cantidad y centro de costo; si falta alguno de esos datos, la solicitud se devuelve para completarla."},
    {"id": "P5", "texto": "Los permisos y certificados son trámites administrativos que atiende administración; la aprobación de gastos mayores corresponde a dirección."},
]

for pasaje in corpus:
    print(f"{pasaje['id']}: {pasaje['texto']}")

P1: Las fallas de acceso, correo o equipos se registran como soporte técnico y las atiende la mesa de ayuda.
P2: Si una solicitud deja a un área sin servicio, se marca con prioridad alta y se atiende antes que las demás.
P3: El estado de una solicitud se consulta en el módulo de seguimiento mediante su código.
P4: Las compras de equipos o suministros se radican con descripción, cantidad y centro de costo; si falta alguno de esos datos, la solicitud se devuelve para completarla.
P5: Los permisos y certificados son trámites administrativos que atiende administración; la aprobación de gastos mayores corresponde a dirección.


## 3. Convertimos el corpus en vectores
Usamos un [modelo multilingüe de Sentence Transformers](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2). `normalize_embeddings=True` deja los vectores con longitud unitaria; así, su producto punto corresponde a la similitud coseno. Los cinco vectores se calculan **una sola vez**, antes de escribir la pregunta.

Mostramos la forma de la matriz y solo los primeros valores de un vector. Los números no se interpretan uno por uno como palabras o reglas.

In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer

NOMBRE_MODELO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
modelo = SentenceTransformer(NOMBRE_MODELO)
textos = [pasaje["texto"] for pasaje in corpus]
vectores_corpus = modelo.encode(textos, convert_to_numpy=True, normalize_embeddings=True)

print("Modelo:", NOMBRE_MODELO)
print("Forma de la matriz (pasajes, dimensiones):", vectores_corpus.shape)
print("Primeros 8 valores de P1:", np.round(vectores_corpus[0, :8], 3))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Forma de la matriz (pasajes, dimensiones): (5, 384)
Primeros 8 valores de P1: [-0.102 -0.114 -0.009 -0.015  0.023  0.032 -0.056  0.025]


## 4. Una búsqueda literal breve
El término `computador` aparece en la pregunta que haremos, pero no en nuestro corpus (allí se habla de `equipos`). Una coincidencia literal con esa palabra no encontrará ningún pasaje. Esto prepara la comparación con la búsqueda semántica.

In [4]:
termino = "computador"
coincidencias = [p["id"] for p in corpus if termino in p["texto"].casefold()]
print(f"Pasajes que contienen literalmente '{termino}': {coincidencias}")

Pasajes que contienen literalmente 'computador': []


## 5. Escribe aquí la pregunta del usuario
**Esta es la celda donde ingresamos el texto del usuario.** En Colab se puede editar el campo `CONSULTA` del formulario o cambiar directamente el texto entre comillas y ejecutar de nuevo. Predicción antes de ejecutar: ¿esperamos ver P1 entre los primeros resultados?

In [5]:
CONSULTA = "Mi computador no enciende y no puedo abrir mi correo; ¿quién me ayuda?" #@param {type:"string"}
K = 3 #@param {type:"integer"}
print("Consulta recibida:", CONSULTA)

Consulta recibida: Mi computador no enciende y no puedo abrir mi correo; ¿quién me ayuda?


## 6. Buscamos los pasajes más cercanos
Convertimos la consulta con el **mismo modelo**, calculamos la similitud coseno y ordenamos los resultados. `K` indica cuántos pasajes mostramos. El puntaje permite ordenar; no representa un porcentaje de certeza.

In [6]:
def buscar(consulta, k=3):
    if not consulta.strip():
        raise ValueError("Escribe una consulta antes de buscar.")
    if not 1 <= k <= len(corpus):
        raise ValueError(f"K debe estar entre 1 y {len(corpus)}.")

    vector_consulta = modelo.encode(consulta, convert_to_numpy=True, normalize_embeddings=True)
    puntajes = vectores_corpus @ vector_consulta
    indices = np.argsort(-puntajes)[:k]
    return [(corpus[int(i)]["id"], float(puntajes[i]), corpus[int(i)]["texto"]) for i in indices]

resultados = buscar(CONSULTA, K)
for puesto, (identificador, similitud, texto) in enumerate(resultados, 1):
    print(f"{puesto}. {identificador} | similitud={similitud:.3f} | {texto}")

assert all(resultados[i][1] >= resultados[i + 1][1] for i in range(len(resultados) - 1))

1. P1 | similitud=0.357 | Las fallas de acceso, correo o equipos se registran como soporte técnico y las atiende la mesa de ayuda.
2. P2 | similitud=0.148 | Si una solicitud deja a un área sin servicio, se marca con prioridad alta y se atiende antes que las demás.
3. P5 | similitud=0.095 | Los permisos y certificados son trámites administrativos que atiende administración; la aprobación de gastos mayores corresponde a dirección.


## 7. La prueba importante: resultado cercano, información insuficiente
Ejecutamos dos preguntas que el corpus **no puede responder por completo**. Debemos leer el texto recuperado antes de decidir qué decir al usuario. No fijamos un umbral automático con cinco ejemplos: un primer lugar siempre existe, incluso si falta la respuesta.

In [7]:
for pregunta in ["¿Cuál es el horario de atención de la mesa de ayuda?", "¿Cuál es el estado actual de SOL-1001?"]:
    print("\nPregunta:", pregunta)
    for identificador, similitud, texto in buscar(pregunta, k=2):
        print(f"  {identificador} | {similitud:.3f} | {texto}")


Pregunta: ¿Cuál es el horario de atención de la mesa de ayuda?
  P1 | 0.402 | Las fallas de acceso, correo o equipos se registran como soporte técnico y las atiende la mesa de ayuda.
  P2 | 0.315 | Si una solicitud deja a un área sin servicio, se marca con prioridad alta y se atiende antes que las demás.

Pregunta: ¿Cuál es el estado actual de SOL-1001?
  P1 | 0.050 | Las fallas de acceso, correo o equipos se registran como soporte técnico y las atiende la mesa de ayuda.
  P3 | 0.037 | El estado de una solicitud se consulta en el módulo de seguimiento mediante su código.


### Preguntas para cerrar la demostración
1. ¿Cuál fue el primer pasaje ante la pregunta por el computador y el correo? ¿Coincidieron las palabras o el significado?
2. ¿Algún pasaje indica el horario de atención de la mesa de ayuda?
3. P3 explica dónde consultar una solicitud. ¿Contiene el estado actual de `SOL-1001`? En la Clase 04 consultábamos ese dato mediante una herramienta; aquí solo buscamos textos.

## Siguiente paso en el proyecto
Sustituyan `corpus` por unos pocos pasajes **autorizados y verificables** de su proyecto; conserven identificadores, procedencia y fecha de consulta. Ejecuten una paráfrasis, una coincidencia exacta y una pregunta fuera de cobertura. Registren en la bitácora el modelo, `K`, los pasajes recuperados y un fallo observado. La preparación e indexación de documentos completos corresponde a la Clase 07.

Documentación: [Sentence Transformers — Semantic Search](https://www.sbert.net/examples/sentence_transformer/applications/semantic-search/README.html).